# Column Profiling EDA — Freddie Mac SFLLD

Purpose-built to answer one question: **which columns should `config.features.categorical_features` actually contain, and which columns are constant/near-constant/ID-like and should be excluded from modeling?**

This directly follows from a real bug hit in production: the models' auto-detection heuristic (`identify_categorical_columns` in `models/dask_utils.py`, "numeric column with <20 unique values ⇒ categorical") swept **88 columns** into categorical encoding instead of the intended ~14, and a handful of genuinely single-valued flags (e.g. `SUPER_CONFORMING_FLAG`) became literal constant columns after encoding — which crashed `dask-glm`'s logistic regression with `Multiple constant columns detected!`.

This notebook profiles the real feature dataset with Dask (lazy, out-of-core — never loads the 47M-row panel into memory) and produces:

1. A dtype-based categorical/numeric split (ground truth, no heuristics)
2. Cardinality, null-rate, and constant/near-constant flags for every column, computed via bounded column-wise reductions
3. A diff between the heuristic's guess and `config`'s current explicit list
4. Ready-to-paste `config/config.py` list literals for `categorical_features` and a new `excluded_features` (constant/ID-like) list

**How to use the output:** paste the printed lists into `config/config.py`. No model code changes are needed beyond what's already there — `main.py` already passes `config.features.categorical_features` explicitly into every model constructor (`categorical_columns=` / `cat_features=`), so editing the config list is sufficient. For the constant/ID-like columns, add them to `FeatureConfig.drop_features` (also in config.py) so they're excluded from the design matrix entirely, upstream of any model.

## 1. Setup

In [1]:
import sys, os
# Adjust if this notebook lives somewhere other than the project root.
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import dask.dataframe as dd

from config.config import config
from models.dask_utils import identify_categorical_columns

paths = config['paths']
features_cfg = config['features']

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)


## 2. Load the feature dataset lazily

Prefers `train_data.parquet` (post train/val/test split) if it already exists, since that's exactly what gets handed to the models; falls back to `feature_dataset.parquet` (pre-split, pre-target) otherwise. Reading via `dd.read_parquet` is lazy — nothing is pulled into memory by this cell.

In [2]:
candidate_paths = [
    ("train_data", paths.train_data),
    ("feature_dataset", paths.feature_dataset),
]

ddf = None
source_used = None
for name, p in candidate_paths:
    try:
        ddf = dd.read_parquet(p)
        source_used = name
        print(f"Loaded '{name}' from: {p}")
        break
    except Exception as e:
        print(f"Could not read '{name}' ({p}): {e}")

if ddf is None:
    raise FileNotFoundError(
        "No parquet dataset found at any candidate path. Run the data-ingestion "
        "-> cleaning -> feature-engineering -> target-creation -> dataset-creation "
        "modules first (or point `candidate_paths` above at an existing file)."
    )

print(f"\nPartitions: {ddf.npartitions}")
print(f"Columns:    {len(ddf.columns)}")


Loaded 'train_data' from: D:/Projects/credit_risk_scoring/data/features/train_data.parquet

Partitions: 128
Columns:    122


## 3. Row count (one full pass)

This is the only cell in the notebook that scans every row — it's a single `count()` reduction (no shuffle, no full materialization), but on the full 47M-row panel it can still take a few minutes. Skip this cell if you just want the column-level profile below; nothing after this depends on it.

In [3]:
n_rows = len(ddf)  # triggers a single count() pass across all partitions
print(f"Rows: {n_rows:,}")


Rows: 3,528,464


## 4. Dtype-based split (ground truth — no heuristics, no computation)

Columns are genuinely categorical (string/object) or genuinely numeric by construction of the Spark feature pipeline — this is just reading Dask's tracked metadata, not inferring anything.

In [4]:
dtype_categorical = [c for c, dt in ddf.dtypes.items()
                     if not pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_datetime64_any_dtype(dt)]
dtype_numeric = [c for c in ddf.columns if c not in dtype_categorical]

print(f"String/object dtype columns ({len(dtype_categorical)}):")
print(dtype_categorical)
print(f"\nNumeric dtype columns ({len(dtype_numeric)}):")
print(dtype_numeric)


String/object dtype columns (34):
['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD', 'CURRENT_LOAN_DELINQUENCY_STATUS', 'DEFECT_SETTLEMENT_DATE', 'MODIFICATION_FLAG', 'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 'DDLPI', 'INTEREST_RATE_STEP_INDICATOR', 'PAYMENT_DEFERRAL_FLAG', 'DELINQUENCY_DUE_TO_DISASTER', 'BORROWER_ASSISTANCE_STATUS_CODE', 'FIRST_PAYMENT_DATE', 'FIRST_TIME_HOMEBUYER_FLAG', 'MATURITY_DATE', 'MSA', 'OCCUPANCY_STATUS', 'CHANNEL', 'PPM_FLAG', 'AMORTIZATION_TYPE', 'PROPERTY_STATE', 'PROPERTY_TYPE', 'POSTAL_CODE', 'LOAN_PURPOSE', 'SELLER_NAME', 'SERVICER_NAME', 'SUPER_CONFORMING_FLAG', 'PRE_RELIEF_REFINANCE_LSN', 'SPECIAL_ELIGIBILITY_PROGRAM', 'RELIEF_REFINANCE_INDICATOR', 'IO_INDICATOR', 'MI_CANCELLATION_INDICATOR', 'last_delinquent_month', 'last_modification_month']

Numeric dtype columns (88):
['CURRENT_ACTUAL_UPB', 'LOAN_AGE', 'REMAINING_MONTHS_TO_LEGAL_MATURITY', 'CURRENT_INTEREST_RATE', 'CURRENT_NON_INTEREST_BEARING_UPB', 'MI_RECOVERIES', 'NET_SALE_PROCEEDS'

## 5. Cardinality per column (single-pass approximate `nunique`)

Uses Dask's `nunique_approx()` (HyperLogLog-based), which is a single bounded pass over the data — not a full shuffle/groupby like exact `nunique()` would require at 47M rows. Approximate counts are within a few percent, which is more than accurate enough for a categorical/numeric decision.

In [5]:
cardinality = {}
for c in ddf.columns:
    try:
        cardinality[c] = int(ddf[c].nunique_approx().compute())
    except Exception as e:
        cardinality[c] = None
        print(f"  [skipped] {c}: {e}")

cardinality_s = pd.Series(cardinality, name="approx_cardinality").sort_values()
cardinality_s.to_frame()


,approx_cardinality
DEFECT_SETTLEMENT_DATE,1
MI_RECOVERIES,1
NON_MI_RECOVERIES,1
NET_SALE_PROCEEDS,1
DDLPI,1
ZERO_BALANCE_CODE,1
ZERO_BALANCE_EFFECTIVE_DATE,1
BORROWER_ASSISTANCE_STATUS_CODE,1
PAYMENT_DEFERRAL_FLAG,1
CUMULATIVE_MODIFICATION_COST,1


## 6. Null rate per column (single reduction pass)

`isnull().mean()` is a single column-wise reduction across all partitions — bounded output (one fraction per column), never a row-wise materialization.

In [6]:
null_rate = ddf.isnull().mean().compute()
null_rate.name = "null_rate"
null_rate.sort_values(ascending=False).to_frame()


,null_rate
ZERO_BALANCE_EFFECTIVE_DATE,1.000000
ZERO_BALANCE_CODE,1.000000
DEFECT_SETTLEMENT_DATE,1.000000
TAXES_AND_INSURANCE,1.000000
LEGAL_COSTS,1.000000
TOTAL_EXPENSES,1.000000
NET_SALE_PROCEEDS,1.000000
NON_MI_RECOVERIES,1.000000
DDLPI,1.000000
MI_RECOVERIES,1.000000


## 7. Constant / near-constant detection

- **Numeric columns**: `min() == max()` (exact constant) via a single column-wise reduction — the same check now built into `models/logistic.py`.
- **All columns, low-cardinality subset only** (cardinality ≤ 50, from step 5): dominant-value share via `value_counts(normalize=True)`, to catch *near*-constant columns (e.g. 99.7% one value) that technically have >1 unique value but will still behave almost like a constant and contribute little signal while adding collinearity risk. This is intentionally restricted to the low-cardinality subset so it stays cheap — a `value_counts` pass on a genuinely high-cardinality numeric column would be expensive and isn't useful here anyway.

In [7]:
# Exact constant check (numeric columns only)
numeric_ddf = ddf[dtype_numeric]
mins = numeric_ddf.min().compute()
maxs = numeric_ddf.max().compute()
is_constant = (mins == maxs)
constant_numeric_cols = is_constant[is_constant].index.tolist()

print(f"Exactly-constant numeric columns ({len(constant_numeric_cols)}):")
print(constant_numeric_cols)


Exactly-constant numeric columns (5):
['PROPERTY_VALUATION_METHOD', 'delinquency_streak_id', 'payment_deferral_count', 'dti_ltv_interaction', 'is_terminated']


In [8]:
# Dominant-value share for low-cardinality columns (categorical OR low-cardinality numeric)
LOW_CARD_THRESHOLD = 50
low_card_cols = [c for c, n in cardinality.items() if n is not None and n <= LOW_CARD_THRESHOLD]

dominant_share = {}
for c in low_card_cols:
    try:
        vc = ddf[c].value_counts(normalize=True, dropna=False).compute()
        dominant_share[c] = float(vc.iloc[0]) if len(vc) else np.nan
    except Exception as e:
        dominant_share[c] = np.nan
        print(f"  [skipped] {c}: {e}")

dominant_share_s = pd.Series(dominant_share, name="dominant_value_share").sort_values(ascending=False)
dominant_share_s.to_frame()


,dominant_value_share
DEFECT_SETTLEMENT_DATE,1.000000e+00
ZERO_BALANCE_CODE,1.000000e+00
ZERO_BALANCE_EFFECTIVE_DATE,1.000000e+00
NET_SALE_PROCEEDS,1.000000e+00
MI_RECOVERIES,1.000000e+00
DDLPI,1.000000e+00
TOTAL_EXPENSES,1.000000e+00
LEGAL_COSTS,1.000000e+00
MAINTENANCE_AND_PRESERVATION_COSTS,1.000000e+00
NON_MI_RECOVERIES,1.000000e+00


In [9]:
NEAR_CONSTANT_THRESHOLD = 0.995  # >99.5% of rows share one value
near_constant_cols = dominant_share_s[dominant_share_s > NEAR_CONSTANT_THRESHOLD].index.tolist()
print(f"Near-constant columns (dominant share > {NEAR_CONSTANT_THRESHOLD:.1%}), {len(near_constant_cols)}:")
print(near_constant_cols)


Near-constant columns (dominant share > 99.5%), 41:
['DEFECT_SETTLEMENT_DATE', 'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 'NET_SALE_PROCEEDS', 'MI_RECOVERIES', 'DDLPI', 'TOTAL_EXPENSES', 'LEGAL_COSTS', 'MAINTENANCE_AND_PRESERVATION_COSTS', 'NON_MI_RECOVERIES', 'TAXES_AND_INSURANCE', 'CUMULATIVE_MODIFICATION_COST', 'ACTUAL_LOSS_CALCULATION', 'MISCELLANEOUS_EXPENSES', 'dti_ltv_interaction', 'is_terminated', 'payment_deferral_count', 'PAYMENT_DEFERRAL_FLAG', 'DELINQUENT_ACCRUED_INTEREST', 'ZERO_BALANCE_REMOVAL_UPB', 'DELINQUENCY_DUE_TO_DISASTER', 'BORROWER_ASSISTANCE_STATUS_CODE', 'ORIGINAL_LTV', 'SPECIAL_ELIGIBILITY_PROGRAM', 'PROPERTY_VALUATION_METHOD', 'IO_INDICATOR', 'MI_CANCELLATION_INDICATOR', 'RELIEF_REFINANCE_INDICATOR', 'PRE_RELIEF_REFINANCE_LSN', 'AMORTIZATION_TYPE', 'rolling_std_rate_6m', 'rolling_std_balance_6m', 'rolling_std_eltv_6m', 'rate_reduction_after_mod', 'CURRENT_NON_INTEREST_BEARING_UPB', 'ever_modified', 'num_modifications', 'cumulative_delinquency', 'is_de

## 8. High-cardinality columns among string/object columns

Freddie Mac SFLLD has a few columns that are identifier-like rather than categorical in the modeling sense (loan numbers, servicer/seller names, postal codes) — `config.features.drop_features` already excludes the obvious ones (`LOAN_SEQUENCE_NUMBER`, `SELLER_NAME`, `SERVICER_NAME`, `POSTAL_CODE`). This cell double-checks nothing else in the string columns is secretly high-cardinality and would blow up one-hot/ordinal encoding if it slipped into `categorical_features`.

In [10]:
HIGH_CARDINALITY_THRESHOLD = 100  # a "real" categorical column shouldn't need more buckets than this for this dataset

high_card_object_cols = [
    c for c in dtype_categorical
    if cardinality.get(c) is not None and cardinality[c] > HIGH_CARDINALITY_THRESHOLD
]
print(f"High-cardinality string/object columns (>{HIGH_CARDINALITY_THRESHOLD} distinct values), {len(high_card_object_cols)}:")
for c in high_card_object_cols:
    print(f"  {c}: ~{cardinality[c]} distinct values")


High-cardinality string/object columns (>100 distinct values), 7:
  LOAN_SEQUENCE_NUMBER: ~307852 distinct values
  MONTHLY_REPORTING_PERIOD: ~244 distinct values
  FIRST_PAYMENT_DATE: ~144 distinct values
  MATURITY_DATE: ~411 distinct values
  MSA: ~454 distinct values
  POSTAL_CODE: ~892 distinct values
  last_delinquent_month: ~158 distinct values


## 9. Compare the model's auto-detection heuristic against `config`'s current list

This reproduces the exact discrepancy that caused the earlier production bug: `identify_categorical_columns()` (the fallback used whenever a model isn't given an explicit list) vs. `config.features.categorical_features` (the explicit list `main.py` now passes to every model).

In [11]:
heuristic_categorical = identify_categorical_columns(ddf)
config_categorical = list(features_cfg.categorical_features)

only_in_heuristic = sorted(set(heuristic_categorical) - set(config_categorical))
only_in_config = sorted(set(config_categorical) - set(heuristic_categorical))

print(f"Heuristic flagged {len(heuristic_categorical)} columns as categorical.")
print(f"Config currently lists {len(config_categorical)} columns as categorical.")
print(f"\nFlagged by heuristic but NOT in config ({len(only_in_heuristic)}):")
print(only_in_heuristic)
print(f"\nIn config but NOT flagged by heuristic ({len(only_in_config)}):")
print(only_in_config)


Heuristic flagged 92 columns as categorical.
Config currently lists 14 columns as categorical.

Flagged by heuristic but NOT in config (78):
['ACTUAL_LOSS_CALCULATION', 'CUMULATIVE_MODIFICATION_COST', 'CURRENT_MONTH_MODIFICATION_COST', 'CURRENT_NON_INTEREST_BEARING_UPB', 'DDLPI', 'DEFECT_SETTLEMENT_DATE', 'DELINQUENT_ACCRUED_INTEREST', 'ELTV', 'FIRST_PAYMENT_DATE', 'INTEREST_RATE_STEP_INDICATOR', 'IO_INDICATOR', 'LEGAL_COSTS', 'LOAN_SEQUENCE_NUMBER', 'MAINTENANCE_AND_PRESERVATION_COSTS', 'MATURITY_DATE', 'MISCELLANEOUS_EXPENSES', 'MI_CANCELLATION_INDICATOR', 'MI_PERCENTAGE', 'MI_RECOVERIES', 'MONTHLY_REPORTING_PERIOD', 'MSA', 'NET_SALE_PROCEEDS', 'NON_MI_RECOVERIES', 'NUMBER_OF_BORROWERS', 'NUMBER_OF_UNITS', 'ORIGINAL_LTV', 'POSTAL_CODE', 'PPM_FLAG', 'PRE_RELIEF_REFINANCE_LSN', 'PROPERTY_VALUATION_METHOD', 'SELLER_NAME', 'SERVICER_NAME', 'SPECIAL_ELIGIBILITY_PROGRAM', 'TAXES_AND_INSURANCE', 'TOTAL_EXPENSES', 'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 'ZERO_BALANCE_REMOVAL_UPB'

## 10. Consolidated column profile

One table, one row per column, everything above joined together — this is what you actually want to scan to make categorical/numeric/drop decisions.

In [12]:
profile = pd.DataFrame({"column": ddf.columns}).set_index("column")
profile["dtype"] = pd.Series({c: str(dt) for c, dt in ddf.dtypes.items()})
profile["is_object_dtype"] = profile.index.isin(dtype_categorical)
profile["approx_cardinality"] = cardinality_s
profile["null_rate"] = null_rate
profile["is_constant_numeric"] = profile.index.isin(constant_numeric_cols)
profile["dominant_value_share"] = dominant_share_s
profile["is_near_constant"] = profile.index.isin(near_constant_cols)
profile["in_config_categorical_features"] = profile.index.isin(config_categorical)
profile["flagged_by_heuristic"] = profile.index.isin(heuristic_categorical)
profile["is_high_cardinality_object"] = profile.index.isin(high_card_object_cols)
profile["in_config_drop_features"] = profile.index.isin(features_cfg.drop_features)

profile = profile.sort_values(["is_object_dtype", "approx_cardinality"], ascending=[False, True])
profile


,dtype,is_object_dtype,approx_cardinality,null_rate,is_constant_numeric,dominant_value_share,is_near_constant,in_config_categorical_features,flagged_by_heuristic,is_high_cardinality_object,in_config_drop_features
column,,,,,,,,,,,
DEFECT_SETTLEMENT_DATE,string,True,1,1.000000,False,1.000000e+00,True,False,True,False,False
ZERO_BALANCE_CODE,string,True,1,1.000000,False,1.000000e+00,True,False,True,False,False
ZERO_BALANCE_EFFECTIVE_DATE,string,True,1,1.000000,False,1.000000e+00,True,False,True,False,False
DDLPI,string,True,1,1.000000,False,1.000000e+00,True,False,True,False,False
PAYMENT_DEFERRAL_FLAG,string,True,1,1.000000,False,1.000000e+00,True,True,True,False,False
DELINQUENCY_DUE_TO_DISASTER,string,True,1,1.000000,False,1.000000e+00,True,True,True,False,False
BORROWER_ASSISTANCE_STATUS_CODE,string,True,1,1.000000,False,1.000000e+00,True,True,True,False,False
AMORTIZATION_TYPE,string,True,1,0.000000,False,1.000000e+00,True,True,True,False,False
PRE_RELIEF_REFINANCE_LSN,string,True,1,1.000000,False,1.000000e+00,True,False,True,False,False


## 11. Columns that need a manual decision

Everything with `is_object_dtype == False` and `approx_cardinality` below a threshold is a **numeric** column that happens to have few unique values — this is exactly the set the auto-detection heuristic over-classified as categorical last time. Some of these genuinely ARE categorical in spirit (e.g. a status code stored as an int); most are counts/flags that should stay numeric. Review this list by hand and decide which belong in `categorical_features`.

In [13]:
REVIEW_THRESHOLD = 20  # matches the heuristic's own default threshold

review_candidates = profile[
    (~profile["is_object_dtype"]) & (profile["approx_cardinality"] <= REVIEW_THRESHOLD)
].copy()

print(f"{len(review_candidates)} numeric, low-cardinality columns to manually review:")
review_candidates[["approx_cardinality", "null_rate", "dominant_value_share", "in_config_categorical_features"]]


49 numeric, low-cardinality columns to manually review:


,approx_cardinality,null_rate,dominant_value_share,in_config_categorical_features
column,,,,
MI_RECOVERIES,1,1.000000,1.000000,False
NET_SALE_PROCEEDS,1,1.000000,1.000000,False
NON_MI_RECOVERIES,1,1.000000,1.000000,False
TOTAL_EXPENSES,1,1.000000,1.000000,False
LEGAL_COSTS,1,1.000000,1.000000,False
MAINTENANCE_AND_PRESERVATION_COSTS,1,1.000000,1.000000,False
TAXES_AND_INSURANCE,1,1.000000,1.000000,False
MISCELLANEOUS_EXPENSES,1,1.000000,1.000000,False
ACTUAL_LOSS_CALCULATION,1,1.000000,1.000000,False


For each of these, a quick look at the *actual* distinct values (not just the count) makes the categorical-vs-numeric call easier — e.g. `observation_month` taking values 1–12 is clearly ordinal/numeric, while a status code taking values like `{0, 1, 2, 3, 6, 9, 'R'}` is clearly categorical.

In [14]:
for c in review_candidates.index:
    try:
        sample_vals = ddf[c].dropna().unique().compute()
        sample_vals = sorted(sample_vals.tolist())[:15]
    except Exception as e:
        sample_vals = f"<could not sample: {e}>"
    print(f"{c:45s} n~{cardinality.get(c)!s:>4}  values: {sample_vals}")


MI_RECOVERIES                                 n~   1  values: []
NET_SALE_PROCEEDS                             n~   1  values: []
NON_MI_RECOVERIES                             n~   1  values: []
TOTAL_EXPENSES                                n~   1  values: []
LEGAL_COSTS                                   n~   1  values: []
MAINTENANCE_AND_PRESERVATION_COSTS            n~   1  values: []
TAXES_AND_INSURANCE                           n~   1  values: []
MISCELLANEOUS_EXPENSES                        n~   1  values: []
ACTUAL_LOSS_CALCULATION                       n~   1  values: []
CUMULATIVE_MODIFICATION_COST                  n~   1  values: []
ZERO_BALANCE_REMOVAL_UPB                      n~   1  values: []
DELINQUENT_ACCRUED_INTEREST                   n~   1  values: []
ORIGINAL_LTV                                  n~   1  values: []
PROPERTY_VALUATION_METHOD                     n~   1  values: [7]
payment_deferral_count                        n~   1  values: [0]
rolling_std_balance_6m 

## 12. Save the full profile to disk

In [15]:
out_dir = paths.eda_dir
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "column_profile.csv")
profile.to_csv(out_path)
print(f"Saved column profile to: {out_path}")


Saved column profile to: D:/Projects/credit_risk_scoring/results/eda\column_profile.csv


## 13. Ready-to-paste `config/config.py` snippets

Two lists, printed as valid Python so you can paste them straight into `FeatureConfig` in `config/config.py`:

- **`categorical_features`**: starts from the dtype-ground-truth string columns (always safe to include) plus anything already in `config`'s current list that also survived the constant/near-constant/high-cardinality checks. It deliberately does **not** auto-add anything from the "needs manual review" list in section 11 — add those yourself after reading the printed sample values above, one at a time, so you don't reintroduce the over-classification bug in the other direction.
- **`excluded_features`** (new list, not yet in config): constant, near-constant, and high-cardinality-object columns — candidates to fold into `FeatureConfig.drop_features` so they're excluded from the design matrix entirely rather than merely relying on each model's constant-column guard.

In [16]:
recommended_categorical = sorted(
    set(dtype_categorical)
    | (set(config_categorical) - set(constant_numeric_cols) - set(near_constant_cols) - set(high_card_object_cols))
)

recommended_excluded = sorted(
    set(constant_numeric_cols) | set(near_constant_cols) | set(high_card_object_cols)
    - set(features_cfg.drop_features)
)

def print_list_literal(name, items):
    print(f"{name}: List[str] = field(default_factory=lambda: [")
    for item in items:
        print(f"    '{item}',")
    print("])")
    print()

print_list_literal("categorical_features", recommended_categorical)
print_list_literal("excluded_features  # NEW -- fold into drop_features", recommended_excluded)


categorical_features: List[str] = field(default_factory=lambda: [
    'AMORTIZATION_TYPE',
    'BORROWER_ASSISTANCE_STATUS_CODE',
    'CHANNEL',
    'CURRENT_LOAN_DELINQUENCY_STATUS',
    'DDLPI',
    'DEFECT_SETTLEMENT_DATE',
    'DELINQUENCY_DUE_TO_DISASTER',
    'FIRST_PAYMENT_DATE',
    'FIRST_TIME_HOMEBUYER_FLAG',
    'INTEREST_RATE_STEP_INDICATOR',
    'IO_INDICATOR',
    'LOAN_PURPOSE',
    'LOAN_SEQUENCE_NUMBER',
    'MATURITY_DATE',
    'MI_CANCELLATION_INDICATOR',
    'MODIFICATION_FLAG',
    'MONTHLY_REPORTING_PERIOD',
    'MSA',
    'OCCUPANCY_STATUS',
    'PAYMENT_DEFERRAL_FLAG',
    'POSTAL_CODE',
    'PPM_FLAG',
    'PRE_RELIEF_REFINANCE_LSN',
    'PROPERTY_STATE',
    'PROPERTY_TYPE',
    'RELIEF_REFINANCE_INDICATOR',
    'SELLER_NAME',
    'SERVICER_NAME',
    'SPECIAL_ELIGIBILITY_PROGRAM',
    'SUPER_CONFORMING_FLAG',
    'ZERO_BALANCE_CODE',
    'ZERO_BALANCE_EFFECTIVE_DATE',
    'last_delinquent_month',
    'last_modification_month',
])

excluded_features  # NEW -- 

## 14. What to do with these lists

1. Paste the `categorical_features` list into `FeatureConfig.categorical_features` in `config/config.py`, after manually adding any columns you decided were categorical in section 11.
2. Paste the `excluded_features` items into `FeatureConfig.drop_features` in `config/config.py` (merge rather than replace — `drop_features` already has entries like `LOAN_SEQUENCE_NUMBER`, `SELLER_NAME`, etc.).
3. No model code changes are required: `main.py` already passes `config.features.categorical_features` into every model constructor (`categorical_columns=` / `cat_features=`), and `datasets/dataset_creation.py` / feature selection already respects `drop_features`. Re-run `python main.py --module dataset_creation` (or the relevant module) so the updated `drop_features` takes effect in the written Parquet, then re-run training.